# **Wycena opcji europejskich: Model Blacka-Scholesa vs. Symulacja Monte Carlo**

**Maja Kropielnicka** \
**28 kwietnia 2026**

**1) Przygotowanie danych** 

Jako przedmiot analizy wybrano akcję spółki **PKO Bank Polski (PKO BP)** 

**Charakterystyka danych:** 

**Aktualny kurs rynkowy**: $S_t$=96,46 PLN (dane z dnia 28.04.2026) \
**Cena wykonania:** K = 96,50 PLN \
**Termin wygasania:** T=0,5 roku (6 miesięcy) \
**Stopa wolna od ryzyka:** r=0,0375 \
**Zmienność:** $\sigma$ = 0,32

*Dane aktualnego kursu rynkowego pobrałam ze strony [stooq](https://stooq.pl), a stope referencyjną ze strony [NBP](https://www.nbp.pl)

**Zmienność $\sigma$** została oszacowana na podstawie danych historycznych z okresu kończącego się 12.04.2026 r
(w ramach projektu pt. analiza-spolek-bankowych)

Obliczono tam zmienność dzienną dla banku PKO BP: **0.020283**. Aby przejść do zmienności roczej, pomnożono ten wynik przez **$\sqrt{252}$** (pierwiastek z liczby dni handlowych w roku; standardowo przyjmuje się 252 dni).



In [6]:
import numpy as np

zmiennosc_dzienna = 0.020283
zmiennosc_roczna = zmiennosc_dzienna * np.sqrt(252)
print(f"Zmienność roczna: {zmiennosc_roczna:.2f}")

Zmienność roczna: 0.32


**2) Model Blacka-Scholesa**

**Wycena dla opcji call**

Do wyceny opcji zakupu skorzystano z poniższego wzoru:

$$C(S, t) = N(d_1) S_t - N(d_2) PV(K) $$

**Wycena dla opcji put**

Do wyceny opcji sprzedaży skorzystano z poniższego wzoru:

$$P(S, t) = N(-d_2) PV(K) - N(-d_1) S_t$$

gdzie:

$\qquad d_1 = \frac{\ln\left(\frac{S_t}{K}\right) + \left(r + \frac{\sigma^2}{2}\right)(T-t)}{\sigma\sqrt{T-t}} \qquad d_2 = d_1 - \sigma\sqrt{T-t}$ \
<br><br>
$\qquad PV(K)$ -> Present Value (dyskontowanie) $\qquad N(d)$ -> skumulowana funkcja dystrybucji \
$\qquad PV(K) = K e^{-r(T-t)}$

**Dystrybuanta**

Nasza funkcja N(d) (dystrybuanta) określona jest wzorem:

$$N(d) = \frac{1}{\sqrt{2\pi}} \int_{-\infty}^{d} e^{-\frac{x^2}{2}} dx$$

W implementacji numerycznej zastosowano równoważną postać wykorzystującą funkcję błędu `erf`:
$$N(d) = \frac{1}{2} \left[ 1 + \text{erf} \left( \frac{d}{\sqrt{2}} \right) \right]$$

Poniżej implementacja modelu Blacka-Scolesa dla naszych parametrów:


In [7]:
import pandas as pd
import math

# 1. Definicja parametrów
S = 96.46
K = 96.50
T = 0.5 #(T-t)
r = 0.0375
sigma = 0.32

# 2. Funkcja normy
def N(d):
    return 0.5 * (1 + math.erf(d / math.sqrt(2)))

# 3. Implementacja modelu Blacka-Scholesa
def wycena_black_scholes(S, K, T, r, sigma):
    # Obliczamy d1 i d2
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # Cena opcji Call
    call = N(d1) * S - N(d2) * K * np.exp(-r * T)
    # Cena opcji Put
    put = N(-d2) * K * np.exp(-r * T) - N(-d1) * S
    
    return call, put, d1, d2

c_price, p_price, d1_val, d2_val = wycena_black_scholes(S, K, T, r, sigma)

print(f"Wyniki modelu Blacka-Scholesa:")
print(f"-------------------------------")
print(f"Wartość d1: {d1_val:.4f}")
print(f"Wartość d2: {d2_val:.4f}")
print(f"Cena opcji CALL: {c_price:.2f} PLN")
print(f"Cena opcji PUT:  {p_price:.2f} PLN")

Wyniki modelu Blacka-Scholesa:
-------------------------------
Wartość d1: 0.1942
Wartość d2: -0.0321
Cena opcji CALL: 9.51 PLN
Cena opcji PUT:  7.76 PLN


**3) Symulacja Monte Carlo**

Teraz wykorzystamy metode MC poprzez wielokrotne losowanie możliwych ścieżek cenowych akcji w przyszłości (dla 10 000 oraz 50 000 symulacji).

Skorzystamy z modelu geometrycznego ruchu Browna: 

$$S_T = S_0 \cdot \exp\left((r - \frac{\sigma^2}{2})T + \sigma\sqrt{T} \cdot Z\right)$$

który dla każdego losowania oblicza, gdzie cena akcji może być za pół roku

W przypadku **wyceny opcji call**, obliczamy max($S_T-K,0$) : "Gdyby cena skończyła w tym miejscu, to ile bym zarobił?" (jeśli cena jest niższa niż $K$, zarobek wynosi $0$).

Ostatecznie wyciągamy średnią z $n$ scenariuszy (mnożąc przez $e^{-rT}$ -> dyskontowanie): 

$$C \approx e^{-rT} \cdot \frac{1}{n} \sum_{i=1}^{n} \max(S_{T} - K, 0)$$

*dla **opcji put** analogicznie, tylko że bierzemy max($K-S_T,0$) (jeśli cena jest wyższa niż $K$, zarobek wynosi $0$).

Poniżej implementacja kodu:


In [8]:
# 4. Implementacja Monte Carlo
def wycena_monte_carlo(S, K, T, r, sigma, n_simulations):
    # Ustawiamy ziarno losowości, żeby wyniki były powtarzalne
    np.random.seed(42)
    
    # Generujemy losowe zmiany cen (zmienna Z)
    # n_simulations to liczba "scenariuszy" przyszłości
    Z = np.random.standard_normal(n_simulations)
    
    # Obliczamy ceny akcji w momencie wygaśnięcia (T) dla każdego scenariusza (Brown)
    S = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    
    # Obliczamy wypłaty dla opcji Call: max(S - K, 0)
    wyplaty_call = np.maximum(S - K, 0)
    
    # Średnia z wypłat zdyskontowana do dzisiaj (e^-rT)
    cena_mc_call = np.exp(-r * T) * np.mean(wyplaty_call)
    
    # Dla opcji Put: max(K - S, 0)
    wyplaty_put = np.maximum(K - S, 0)
    cena_mc_put = np.exp(-r * T) * np.mean(wyplaty_put)
    
    return cena_mc_call, cena_mc_put

# Obliczenia
mc_call1, mc_put1 = wycena_monte_carlo(S, K, T, r, sigma, n_simulations=10000)

print(f"Wyniki Monte Carlo ({10000} symulacji):")
print(f"--------------------------------------")
print(f"Cena opcji CALL (MC): {mc_call1:.2f} PLN")
print(f"Cena opcji PUT  (MC): {mc_put1:.2f} PLN")
print()

mc_call2, mc_put2 = wycena_monte_carlo(S, K, T, r, sigma, n_simulations=50000)

print(f"Wyniki Monte Carlo ({50000} symulacji):")
print(f"--------------------------------------")
print(f"Cena opcji CALL (MC): {mc_call2:.2f} PLN")
print(f"Cena opcji PUT  (MC): {mc_put2:.2f} PLN")

Wyniki Monte Carlo (10000 symulacji):
--------------------------------------
Cena opcji CALL (MC): 9.52 PLN
Cena opcji PUT  (MC): 7.79 PLN

Wyniki Monte Carlo (50000 symulacji):
--------------------------------------
Cena opcji CALL (MC): 9.51 PLN
Cena opcji PUT  (MC): 7.76 PLN


**4) Porównanie wyników**

Ostatecznie przedstawiamy wszystkie wyniki w tablicy:



In [9]:
#call
abs_err_c1 = abs(c_price - mc_call1)
rel_err_c1 = (abs_err_c1 / c_price) * 100 

abs_err_c2 = abs(c_price - mc_call2)
rel_err_c2 = (abs_err_c2 / c_price) * 100 

#put
abs_err_p1 = abs(p_price - mc_put1)
rel_err_p1 = (abs_err_p1 / p_price) * 100 

abs_err_p2 = abs(p_price - mc_put2)
rel_err_p2 = (abs_err_p2 / p_price) * 100 

dane = [
    {
        'Metoda': 'Black-Scholes',
        'Cena CALL': round(c_price, 4),
        'Błąd Wzgl. CALL (%)': "0.00%",
        'Cena PUT': round(p_price, 4),
        'Błąd Wzgl. PUT (%)': "0.00%"
    },
    {
        'Metoda': 'Monte Carlo (n=10 000)',
        'Cena CALL': round(mc_call1, 4),
        'Błąd Wzgl. CALL (%)': f"{rel_err_c1:.4f}%",
        'Cena PUT': round(mc_put1, 4),
        'Błąd Wzgl. PUT (%)': f"{rel_err_p1:.4f}%"
    },
    {
        'Metoda': 'Monte Carlo (n=50 000)',
        'Cena CALL': round(mc_call2, 4),
        'Błąd Wzgl. CALL (%)': f"{rel_err_c2:.4f}%",
        'Cena PUT': round(mc_put2, 4),
        'Błąd Wzgl. PUT (%)': f"{rel_err_p2:.4f}%"
    }
]

df_kompletny = pd.DataFrame(dane)
df_kompletny

,Metoda,Cena CALL,Błąd Wzgl. CALL (%),Cena PUT,Błąd Wzgl. PUT (%)
0,Black-Scholes,9.5144,0.00%,7.7619,0.00%
1,Monte Carlo (n=10 000),9.5166,0.0230%,7.7932,0.4030%
2,Monte Carlo (n=50 000),9.5066,0.0815%,7.7643,0.0313%


**5) Wnioski**

* Modele działają niemal identycznie: Różnica między wzorem matematycznym (Black-Scholes) a symulacją (Monte Carlo) jest mniejsza niż 0,1%.

* Więcej prób = lepszy wynik: W przypadku opcji Put wyraźnie widać, że zwiększenie liczby symulacji z 10 tys. do 50 tys. "wygładziło" wynik (0.4030% -> 0.0313%)

* Losowość to norma: Niewielkie wahania błędu (jak przy opcji Call) to naturalna cecha metody Monte Carlo. Wynikają one z przypadku przy losowaniu liczb.
